In [1]:
# --- BLOQUE 0: INFRAESTRUCTURA ---
import sys
import gzip
import io
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import requests
from scipy.optimize import curve_fit
from astropy.cosmology import Planck18 as cosmo

print("[SYSTEM] Iniciando Protocolo de Ejecución FINAL (Manual Parsing)...")

# --- BLOQUE 1: DESCARGA ---
FTP_URL = "https://heasarc.gsfc.nasa.gov/FTP/heasarc/dbase/dump/heasarc_fermilgrb.tdat.gz"
print(f"[NET] Descargando: {FTP_URL}")

try:
    response = requests.get(FTP_URL)
    response.raise_for_status()
    with gzip.open(io.BytesIO(response.content), 'rt', encoding='utf-8') as f:
        content = f.read()
except Exception as e:
    print(f"[CRITICAL] Error de descarga: {e}")
    sys.exit()

# --- BLOQUE 2: PARSEO QUIRÚRGICO ---
# El archivo tiene la cabecera separada por ESPACIOS y los datos por PIPES (|).
# Los parsers automáticos fallan aquí. Lo hacemos a mano.

lines = content.split('\n')
header_cols = []
data_start_idx = 0

# 1. Encontrar nombres de columnas (Línea que empieza con 'line[1] =')
for i, line in enumerate(lines):
    if line.startswith('line[1] ='):
        # Quitamos 'line[1] =' y separamos por espacios
        raw_header = line.replace('line[1] =', '').strip()
        header_cols = raw_header.split()
        print(f"[INFO] Cabecera extraída: {len(header_cols)} columnas encontradas.")
        break

if not header_cols:
    print("[ERROR] No se encontró la definición de columnas.")
    sys.exit()

# 2. Encontrar inicio de datos (<DATA>)
for i, line in enumerate(lines):
    if line.strip() == '<DATA>':
        data_start_idx = i + 1
        break

# 3. Cargar datos forzando el separador PIPE (|)
print("[PROC] Leyendo datos con separador PIPE (|)...")
try:
    # Unimos las líneas de datos de nuevo en un string para pandas
    data_content = '\n'.join(lines[data_start_idx:])

    # Usamos el motor de python para ser más tolerantes, header=None porque ya lo tenemos
    df = pd.read_csv(io.StringIO(data_content), sep='|', names=header_cols, header=None, on_bad_lines='skip')

    # Limpieza: HEASARC a veces deja un pipe al final, creando una columna extra vacía
    # Si hay mismatch, pandas puede haber empujado datos.
    # Verificamos columnas clave.

    # Normalizar nombres a mayúsculas
    df.columns = [c.upper() for c in df.columns]
    print(f"[SUCCESS] Tabla cargada. Dimensiones: {df.shape}")

except Exception as e:
    print(f"[CRITICAL] Fallo en lectura manual: {e}")
    sys.exit()

# --- BLOQUE 3: FÍSICA (CON LAS COLUMNAS QUE TENEMOS) ---
# Mapeo explícito basado en tu archivo
col_z = 'REDSHIFT'
col_e = 'EXT_EMISSION_MAX_ENE'    # MeV
col_t = 'EXT_EMISSION_MAX_ENE_T'  # s
col_n = 'NAME'

print(f"[MAP] Usando: Z={col_z}, E={col_e}, T={col_t}")

# Conversión a numérico (coercing errors)
df[col_z] = pd.to_numeric(df[col_z], errors='coerce')
df[col_e] = pd.to_numeric(df[col_e], errors='coerce')
df[col_t] = pd.to_numeric(df[col_t], errors='coerce')

# Filtrado
# 1. Redshift válido
# 2. Energía > 1000 MeV (1 GeV) - Los fotones interesantes
# 3. Tiempo válido
mask = (df[col_z] > 0) & (df[col_e] > 1000) & (df[col_t].notna())

clean_data = df[mask].copy()
clean_data['E_GeV'] = clean_data[col_e] / 1000.0
clean_data['Lags'] = clean_data[col_t] # Usamos el tiempo de llegada como "retraso" relativo al trigger

print(f"[DATA] Eventos LIV Válidos: {len(clean_data)}")

if len(clean_data) < 3:
    print("[WARN] Pocos datos estrictos. Relajando filtro de energía a 100 MeV...")
    mask_relaxed = (df[col_z] > 0) & (df[col_e] > 100) & (df[col_t].notna())
    clean_data = df[mask_relaxed].copy()
    clean_data['E_GeV'] = clean_data[col_e] / 1000.0
    clean_data['Lags'] = clean_data[col_t]
    print(f"[DATA] Eventos recuperados: {len(clean_data)}")

# --- BLOQUE 4: CÁLCULO Y PLOT ---
print("[PHYS] Calculando K-Factors...")

def get_k(z):
    z_grid = np.linspace(0, z, 50)
    Ez = np.sqrt(cosmo.Om0*(1+z_grid)**3 + cosmo.Ode0)
    return np.trapz((1+z_grid)/Ez, z_grid)

if len(clean_data) > 0:
    clean_data['K'] = clean_data[col_z].apply(get_k)

    # Observable LIV: tau_n = dt / E
    # Hipótesis: El retraso escala con la energía.
    clean_data['Lag_Norm'] = clean_data['Lags'] / clean_data['E_GeV']

    X = clean_data['K'].values
    Y = clean_data['Lag_Norm'].values
    C = clean_data['E_GeV'].values

    # Regresión
    def model(k, m, c): return m * k + c

    try:
        # Filtro de outliers extremos para el ajuste (sigma clipping)
        q_low = np.percentile(Y, 10)
        q_high = np.percentile(Y, 90)
        mask_fit = (Y > q_low) & (Y < q_high)

        if np.sum(mask_fit) > 2:
            popt, pcov = curve_fit(model, X[mask_fit], Y[mask_fit])
            m_fit = popt[0]
            m_err = np.sqrt(np.diag(pcov))[0]
        else:
            m_fit = 0
    except: m_fit = 0

    # PLOT
    plt.figure(figsize=(12, 7))
    plt.style.use('dark_background')

    sc = plt.scatter(X, Y, c=C, cmap='rainbow', s=100, alpha=0.8, edgecolors='white', linewidth=0.5)
    plt.colorbar(sc, label='Energía (GeV)')

    if m_fit != 0:
        x_line = np.linspace(0, max(X)*1.1, 10)
        plt.plot(x_line, model(x_line, m_fit, popt[1]), color='cyan', linestyle='--', lw=2, label=f'Pendiente LIV (m={m_fit:.2f})')

    plt.axhline(0, color='white', linestyle=':', alpha=0.6)

    # Etiquetas
    top = clean_data.nlargest(3, 'E_GeV')
    for _, row in top.iterrows():
        plt.annotate(f"{row[col_n]}\n{row['E_GeV']:.1f}G", (row['K'], row['Lag_Norm']), color='yellow', fontsize=9)

    plt.xlabel('Factor K (Distancia)')
    plt.ylabel('Retraso Normalizado (s/GeV)')
    plt.title(f'RESULTADO FINAL: Test de Tiempo de Vuelo\nDatos: {FTP_URL.split("/")[-1]}', color='white')
    plt.legend()
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()

    print(f"\n[RESULT] Pendiente m = {m_fit:.4f}")
else:
    print("[FAIL] No quedaron datos después del filtrado.")

[SYSTEM] Iniciando Protocolo de Ejecución FINAL (Manual Parsing)...
[NET] Descargando: https://heasarc.gsfc.nasa.gov/FTP/heasarc/dbase/dump/heasarc_fermilgrb.tdat.gz
[INFO] Cabecera extraída: 107 columnas encontradas.
[PROC] Leyendo datos con separador PIPE (|)...
[SUCCESS] Tabla cargada. Dimensiones: (232, 107)
[MAP] Usando: Z=REDSHIFT, E=EXT_EMISSION_MAX_ENE, T=EXT_EMISSION_MAX_ENE_T
[DATA] Eventos LIV Válidos: 0
[WARN] Pocos datos estrictos. Relajando filtro de energía a 100 MeV...
[DATA] Eventos recuperados: 0
[PHYS] Calculando K-Factors...
[FAIL] No quedaron datos después del filtrado.
